<a href="https://colab.research.google.com/github/GGSimmons1992/5VfneekyeG9soDiV/blob/main/Notebooks/retrieveMask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# retrieveMask.ipynb
## Goal 1: Extract the final feature mask chosen by the RL agent

In [ ]:
import numpy as np
from stable_baselines3 import DQN
from google.colab import drive

drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/My Drive/Colab Notebooks/SalesReinforcer/Src/')
import salesReinforcerEnvironments

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def eval_feature_env(env, model, n_episodes=20):
  penalized_scores = []
  raw_f1_scores = []
  feature_counts = []

  for _ in range(n_episodes):
      obs, _ = env.reset()
      done = False
      while not done:
          action, _ = model.predict(obs, deterministic=True)
          obs, reward, terminated, truncated, _ = env.step(action)
          done = terminated or truncated

      uw = env.unwrapped

      # Penalized score from current objective
      penalized = uw._calculate_reward()

      # Raw F1 without feature penalty
      old_lambda = uw.feature_penalty_weight
      uw.feature_penalty_weight = 0.0
      raw_f1 = uw._calculate_reward()
      uw.feature_penalty_weight = old_lambda

      n_feats = int(np.sum(uw.feature_mask))

      penalized_scores.append(penalized)
      raw_f1_scores.append(raw_f1)
      feature_counts.append(n_feats)

  return {
      "penalized_mean": float(np.mean(penalized_scores)),
      "raw_f1_mean": float(np.mean(raw_f1_scores)),
      "features_mean": float(np.mean(feature_counts)),
  }

metrics = eval_feature_env(feature_env, model, n_episodes=30)
print(metrics)

In [ ]:
def main():
  full_train_data = dataPrep.retrieveCSVFromDrive("SalesReinforcerTrain.csv")
  train, dev = train_test_split(
      full_train_data,
      test_size=0.2,
      random_state=42,
      stratify=full_train_data["isSubscribed"]
  )

  feature_env.unwrapped.feature_penalty_weight = 0.01
  feature_env.unwrapped.max_steps = 200

  model = DQN(
      "MultiInputPolicy",
      feature_env,
      learning_rate=1e-4,
      buffer_size=200_000,
      learning_starts=5_000,
      batch_size=256,
      gamma=0.995,
      train_freq=4,
      gradient_steps=1,
      target_update_interval=2000,
      exploration_fraction=0.4,
      exploration_final_eps=0.02,
      verbose=1,
      seed=42
  )

  model.learn(total_timesteps=150_000)
  metrics = eval_feature_env(feature_env, model, n_episodes=30)
  print(metrics)

In [ ]:
if __name__ == '__main__':
  main()